# EXPLORACIÓN Y ANÁLISIS DE DATOS

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime

In [2]:
DATA_DIR = Path("./data/raw")
REPORT_DIR = Path("./reports")

In [3]:
FILES = {
    "clientes": DATA_DIR / "clientes.parquet",
    "tarjetas": DATA_DIR / "tarjetas.parquet",
    "transacciones": DATA_DIR / "transacciones.parquet",
    "interacciones_marketing": DATA_DIR / "interacciones_marketing.parquet",
    "catalogo_comercios": DATA_DIR / "catalogo_comercios.parquet",
}

## CARGA DE DATOS

In [5]:
dfs = {}
for name, path in FILES.items():
    dfs[name] = pd.read_parquet(path)
    print(f"### {name}: {dfs[name].shape[0]:,} filas, {dfs[name].shape[1]} columnas")

### clientes: 12,048 filas, 8 columnas
### tarjetas: 16,634 filas, 6 columnas
### transacciones: 194,173 filas, 7 columnas
### interacciones_marketing: 36,000 filas, 6 columnas
### catalogo_comercios: 42 filas, 3 columnas


In [6]:
clientes = dfs.get("clientes")
tarjetas = dfs.get("tarjetas")
transacciones = dfs.get("transacciones")
marketing = dfs.get("interacciones_marketing")
comercios = dfs.get("catalogo_comercios")

## ANÁLISIS GENERAL

In [7]:
def explorar_dataframe(df: pd.DataFrame, nombre: str):
    print(f"\n## Explorando dataframe — {nombre}")
    print(f"- Shape: {df.shape}")
 
    # Tipos de dato
    print(f"- Dtypes:\n{df.dtypes.to_string()}")
 
    # Nulos por columna
    nulos = df.isnull().sum()
    nulos_pct = (nulos / len(df) * 100).round(2)
    tabla_nulos = pd.DataFrame({"nulos": nulos, "pct": nulos_pct})
    tabla_nulos = tabla_nulos[tabla_nulos["nulos"] > 0].sort_values("pct", ascending=False)
    if len(tabla_nulos):
        print(f"- Columnas con nulos:\n{tabla_nulos.to_string()}")
    else:
        print("- Sin nulos en ninguna columna.")
 
    # Duplicados de fila completa
    dup_full = df.duplicated().sum()
    print(f"- Filas 100% duplicadas: {dup_full}")
 
    # Cardinalidad de columnas tipo texto/categoría
    obj_cols = df.select_dtypes(include=["object", "category"]).columns
    for col in obj_cols:
        n_unique = df[col].nunique(dropna=True)
        if n_unique <= 30:
            print(f"  - Valores únicos en '{col}' ({n_unique}): {sorted(df[col].dropna().unique().tolist())}")
        else:
            print(f"  - '{col}': {n_unique} valores únicos (alta cardinalidad, probable ID o texto libre)")


In [8]:
for nombre, df in dfs.items():
    explorar_dataframe(df, nombre)


## Explorando dataframe — clientes
- Shape: (12048, 8)
- Dtypes:
cliente_id           object
cedula               object
nombre_completo      object
fecha_nacimiento     object
ciudad               object
canal_adquisicion    object
estado_cuenta        object
fecha_registro       object
- Columnas con nulos:
                   nulos   pct
canal_adquisicion    212  1.76
- Filas 100% duplicadas: 48
  - 'cliente_id': 12000 valores únicos (alta cardinalidad, probable ID o texto libre)
  - 'cedula': 11940 valores únicos (alta cardinalidad, probable ID o texto libre)
  - 'nombre_completo': 7616 valores únicos (alta cardinalidad, probable ID o texto libre)
  - 'fecha_nacimiento': 10331 valores únicos (alta cardinalidad, probable ID o texto libre)
  - 'ciudad': 45 valores únicos (alta cardinalidad, probable ID o texto libre)
  - Valores únicos en 'canal_adquisicion' (19): ['', ' call_center ', ' organico ', ' publicidad_digital ', ' referido ', 'CALL_CENTER', 'Call_center  ', 'N/A', 'NULL', 

## ANÁLISIS ESPECÍFICO

In [11]:
# QUITAR GUIÓN DE CÉDULAS

clientes['cedula'] = (
        clientes['cedula']
        .astype(str)
        .str.replace('-', "", regex=False)
    )

In [12]:
# VALIDACIÓN MODULO 10
def validar_cedula_ecuador(cedula: str) -> bool:
    """Valida cédula ecuatoriana (10 dígitos) con el algoritmo de módulo 10."""
    if not isinstance(cedula, str) or not cedula.isdigit() or len(cedula) != 10:
        return False

    provincia = int(cedula[0:2])
    if provincia < 1 or provincia > 24:
        return False  

    tercer_digito = int(cedula[2])
    if tercer_digito > 6:
        return False  # para persona natural debe ser 0-6

    coeficientes = [2, 1, 2, 1, 2, 1, 2, 1, 2]
    suma = 0
    for i, coef in enumerate(coeficientes):
        valor = int(cedula[i]) * coef
        if valor >= 10:
            valor -= 9
        suma += valor

    digito_verificador = int(cedula[9])
    return (10 - (suma % 10)) % 10 == digito_verificador


# aplicarlo a las cédulas duplicadas
clientes["cedula_valida"] = clientes["cedula"].astype(str).apply(validar_cedula_ecuador)
print(clientes.groupby("cedula")["cedula_valida"].first().value_counts())

cedula_valida
False    11108
True       832
Name: count, dtype: int64


### ESTANDARIZACIÓN DE FECHAS

In [ ]:
FORMATOS_CANDIDATOS = [
    "%Y-%m-%dT%H:%M:%S",   
    "%Y-%m-%d %H:%M:%S",
    "%Y-%m-%d",            
    "%d/%m/%Y %H:%M",
    "%d/%m/%Y",            
    "%d-%m-%Y",
    "%d.%m.%Y",
    "%m/%d/%Y",            
    "%Y%m%d",
]

def estandarizar_fecha(
    serie: pd.Series,
    nombre_columna: str = "fecha",
    fecha_min: pd.Timestamp = None,
    fecha_max: pd.Timestamp = None,
) -> pd.DataFrame:
    """
    Recibe una serie de fechas en formatos mixtos (string y/o numérico tipo
    serial de Excel o epoch Unix) y devuelve un DataFrame con:
      - fecha_estandarizada: datetime64, o NaT si no se pudo parsear
      - formato_detectado: qué patrón matcheó (para auditoría)
      - fecha_valida: bool
 
    fecha_min / fecha_max acotan qué se considera "plausible" al interpretar
    valores NUMÉRICOS (serial de Excel / epoch Unix) — importante porque el
    rango razonable depende del significado de la columna:
      - fecha_nacimiento: puede ser pre-1970 (epoch negativo), ej. fecha_min=1900-01-01
      - fecha_registro/emision/transaccion: acotado al periodo de operación
        del negocio, ej. fecha_min=2015-01-01
    """
    if fecha_min is None:
        fecha_min = pd.Timestamp("1900-01-01")
    if fecha_max is None:
        fecha_max = pd.Timestamp.now() + pd.Timedelta(weeks=52)
 
    epoch_ref = pd.Timestamp("1970-01-01")
    epoch_min_s = (fecha_min - epoch_ref) // pd.Timedelta(seconds=1)
    epoch_max_s = (fecha_max - epoch_ref) // pd.Timedelta(seconds=1)
 
    excel_ref = pd.Timestamp("1899-12-30")
    serial_min = (fecha_min - excel_ref).days
    serial_max = (fecha_max - excel_ref).days
 
    serie_str = serie.astype(str).str.strip()
    resultado = pd.Series(pd.NaT, index=serie.index, dtype="datetime64[ns]")
    formato_detectado = pd.Series(pd.NA, index=serie.index, dtype="object")
 
    # 1. Intentar cada formato explícito, solo sobre lo que sigue pendiente
    pendiente = resultado.isna() & serie.notna()
    for fmt in FORMATOS_CANDIDATOS:
        if not pendiente.any():
            break
        parseado = pd.to_datetime(serie_str[pendiente], format=fmt, errors="coerce")
        exito = parseado.notna()
        idx_exito = parseado[exito].index
        resultado.loc[idx_exito] = parseado[exito]
        formato_detectado.loc[idx_exito] = fmt
        pendiente = resultado.isna() & serie.notna()
 
    # 2. Lo que sigue pendiente: probar como serial de Excel (numérico,
    #    días desde 1899-12-30) — común cuando la fecha se exportó desde
    #    una planilla y perdió el formato de fecha en el camino.
    if pendiente.any():
        numericos = pd.to_numeric(serie[pendiente], errors="coerce")
        es_serial = numericos.notna() & numericos.between(serial_min, serial_max)
        if es_serial.any():
            idx_serial = numericos[es_serial].index
            fechas_excel = pd.to_datetime(numericos[es_serial], unit="D", origin="1899-12-30")
            resultado.loc[idx_serial] = fechas_excel
            formato_detectado.loc[idx_serial] = "excel_serial"
        pendiente = resultado.isna() & serie.notna()
 
    # 3. Lo que sigue pendiente: probar como timestamp Unix epoch, en
    #    segundos o milisegundos. Los rangos de magnitud (dentro de
    #    [fecha_min, fecha_max]) no se solapan entre sí ni con el serial de
    #    Excel, así que se distinguen con seguridad por orden de magnitud.
    #    Nota: epoch_min_s puede ser NEGATIVO si fecha_min es anterior a
    #    1970 (ej. fecha_nacimiento) — eso es intencional y correcto.
    if pendiente.any():
        numericos = pd.to_numeric(serie[pendiente], errors="coerce")
 
        es_epoch_s = numericos.notna() & numericos.between(epoch_min_s, epoch_max_s)
        if es_epoch_s.any():
            idx_s = numericos[es_epoch_s].index
            resultado.loc[idx_s] = pd.to_datetime(numericos[es_epoch_s], unit="s")
            formato_detectado.loc[idx_s] = "unix_epoch_segundos"
        pendiente = resultado.isna() & serie.notna()
 
        es_epoch_ms = numericos.notna() & numericos.between(epoch_min_s * 1000, epoch_max_s * 1000)
        if es_epoch_ms.any():
            idx_ms = numericos[es_epoch_ms].index
            resultado.loc[idx_ms] = pd.to_datetime(numericos[es_epoch_ms], unit="ms")
            formato_detectado.loc[idx_ms] = "unix_epoch_milisegundos"
 
    fecha_valida = resultado.notna()
 
    return pd.DataFrame({
        f"{nombre_columna}_estandarizada": resultado,
        f"{nombre_columna}_formato_detectado": formato_detectado,
        f"{nombre_columna}_valida": fecha_valida,
    })
 
 
def reportar_estandarizacion_fechas(df: pd.DataFrame, col_fecha: str, nombre_tabla: str):
    """Aplica estandarizar_fecha y loguea el resumen de formatos encontrados."""
    resultado = estandarizar_fecha(df[col_fecha], nombre_columna=col_fecha)
    print(f"\n### Estandarización de '{col_fecha}' en {nombre_tabla}")
    print(f"- Formatos detectados:\n{resultado[f'{col_fecha}_formato_detectado'].value_counts(dropna=False).to_string()}")
    n_invalidas = (~resultado[f"{col_fecha}_valida"] & df[col_fecha].notna()).sum()
    if n_invalidas > 0:
        print(f"{n_invalidas} valores de '{col_fecha}' no calzaron con ningún formato conocido — revisar a mano.")
    return resultado

In [10]:
for nombre, df in dfs.items():
    date_cols = [c for c in df.columns if "fecha" in c.lower() or "date" in c.lower()]
    for col in date_cols:
        reportar_estandarizacion_fechas(df, col, nombre)


### Estandarización de 'fecha_nacimiento' en clientes
- Formatos detectados:
fecha_nacimiento_formato_detectado
%Y-%m-%d                   8429
%d/%m/%Y                   3018
unix_epoch_milisegundos     601

### Estandarización de 'fecha_registro' en clientes
- Formatos detectados:
fecha_registro_formato_detectado
%Y-%m-%d                   8435
%d/%m/%Y                   3010
unix_epoch_milisegundos     603

### Estandarización de 'fecha_emision' en tarjetas
- Formatos detectados:
fecha_emision_formato_detectado
%Y-%m-%d                   11638
%d/%m/%Y                    4153
unix_epoch_milisegundos      835
<NA>                           8

### Estandarización de 'fecha_activacion' en tarjetas
- Formatos detectados:
fecha_activacion_formato_detectado
%Y-%m-%d                   9697
%d/%m/%Y                   3427
<NA>                       2817
unix_epoch_milisegundos     693

### Estandarización de 'fecha' en transacciones
- Formatos detectados:
fecha_formato_detectado
%Y-%m-%d  

In [28]:
# VERIFICACION VALORES NULOS comercio_codigo
tabla_cruzada = pd.crosstab(transacciones["tipo_transaccion"], transacciones["comercio_codigo"].isna())
tabla_cruzada.columns = ["tiene_comercio_codigo", "comercio_codigo_nulo"]
print(tabla_cruzada)

# % de nulos dentro de cada tipo de transacción
print((transacciones.groupby("tipo_transaccion")["comercio_codigo"]
       .apply(lambda x: x.isna().mean() * 100)
       .rename("pct_nulos")))

                  tiene_comercio_codigo  comercio_codigo_nulo
tipo_transaccion                                             
 cash_in                              0                  3412
 cash_out                             0                  2360
 compra_tarjeta                    1337                     0
 p2p_in                               0                  1687
 p2p_out                              0                  1727
 pago_servicio                        0                   561
 recarga_celular                      0                  1116
 remesa                               0                   609
CASH_IN                               0                  3591
CASH_OUT                              0                  2366
COMPRA_TARJETA                     1341                     0
Cash In                               0                  3471
Cash Out                              0                  2314
Compra Tarjeta                     1260                     0
P2P In  

In [29]:

def chequear_huerfanos(df_hijo, col_hijo, df_maestro, col_maestro, nombre_hijo, nombre_maestro):
    if df_hijo is None or df_maestro is None:
        return
    if col_hijo not in df_hijo.columns or col_maestro not in df_maestro.columns:
        print(f"No se pudo chequear {nombre_hijo} -> {nombre_maestro}: columna no encontrada.")
        return
    ids_maestro = set(df_maestro[col_maestro].dropna())
    ids_hijo = set(df_hijo[col_hijo].dropna())
    huerfanos = ids_hijo - ids_maestro
    pct = len(huerfanos) / len(ids_hijo) * 100 if ids_hijo else 0
    log(f"- {nombre_hijo}.{col_hijo} sin match en {nombre_maestro}.{col_maestro}: "
        f"{len(huerfanos)} IDs únicos ({pct:.2f}% de los IDs presentes en {nombre_hijo})")
 
 
print("\n## Integridad referencial")
 
id_cliente_maestro = "cliente_id" if clientes is not None and "cliente_id" in clientes.columns else None
if id_cliente_maestro:
    id_col_tarjetas = next((c for c in (tarjetas.columns if tarjetas is not None else []) if "cliente" in c.lower()), None)
    id_col_tx = next((c for c in (transacciones.columns if transacciones is not None else []) if "cliente" in c.lower()), None)
    id_col_mkt = next((c for c in (marketing.columns if marketing is not None else []) if "cliente" in c.lower()), None)
 
    if id_col_tarjetas:
        chequear_huerfanos(tarjetas, id_col_tarjetas, clientes, id_cliente_maestro, "tarjetas", "clientes")
    if id_col_tx:
        chequear_huerfanos(transacciones, id_col_tx, clientes, id_cliente_maestro, "transacciones", "clientes")
    if id_col_mkt:
        chequear_huerfanos(marketing, id_col_mkt, clientes, id_cliente_maestro, "interacciones_marketing", "clientes")
 
# comercios <-> transacciones (código de comercio en compras con tarjeta)
if comercios is not None and transacciones is not None:
    col_codigo_comercio = next((c for c in comercios.columns if "codigo" in c.lower() or "code" in c.lower()), None)
    col_codigo_tx = next((c for c in transacciones.columns if "comercio" in c.lower()), None)
    if col_codigo_comercio and col_codigo_tx:
        chequear_huerfanos(transacciones, col_codigo_tx, comercios, col_codigo_comercio,
                            "transacciones", "catalogo_comercios")
 


## Integridad referencial
- tarjetas.cliente_id sin match en clientes.cliente_id: 248 IDs únicos (2.12% de los IDs presentes en tarjetas)
- transacciones.cliente_id sin match en clientes.cliente_id: 1531 IDs únicos (11.32% de los IDs presentes en transacciones)
- interacciones_marketing.cliente_id sin match en clientes.cliente_id: 360 IDs únicos (3.06% de los IDs presentes en interacciones_marketing)
- transacciones.comercio_codigo sin match en catalogo_comercios.comercio_codigo: 2 IDs únicos (4.55% de los IDs presentes en transacciones)
